In [1]:
#123

%pip install -q duckdb pyarrow pandas tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import io
import os
import re
import sys
import json
import math
import shutil
import zipfile
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import duckdb
import pandas as pd
from tqdm.auto import tqdm

In [3]:
# Root folder that contains the Freddie Mac yearly ZIPs
DATA_ROOT = Path(r"C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data")

# Output folders
PARQUET_ROOT = DATA_ROOT / "parquet"
ORIGINATION_OUT = PARQUET_ROOT / "origination"
PERFORMANCE_OUT = PARQUET_ROOT / "performance"

# Temporary extraction folder
TEMP_ROOT = DATA_ROOT / "_tmp_ingest"
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# DuckDB database file for the ingestion session
DUCKDB_PATH = DATA_ROOT / "freddie_mac_ingest.duckdb"

# Behavior
OVERWRITE_EXISTING = False
MAX_THREADS = max(1, (os.cpu_count() or 4) - 1)
DUCKDB_MEMORY_LIMIT = "24GB"   # adjust if needed
PARQUET_COMPRESSION = "ZSTD"
PARQUET_ROW_GROUP_SIZE = 250_000

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"DUCKDB_PATH: {DUCKDB_PATH}")
print(f"MAX_THREADS: {MAX_THREADS}")

DATA_ROOT: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data
DUCKDB_PATH: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data\freddie_mac_ingest.duckdb
MAX_THREADS: 7


In [4]:
ORIGINATION_COLS = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_flag",
    "maturity_date",
    "msa_or_metropolitan_division",
    "mortgage_insurance_percentage",
    "number_of_units",
    "occupancy_status",
    "original_combined_loan_to_value",
    "original_debt_to_income_ratio",
    "original_upb",
    "original_loan_to_value",
    "original_interest_rate",
    "channel",
    "prepayment_penalty_mortgage_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_sequence_number",
    "loan_purpose",
    "original_loan_term",
    "number_of_borrowers",
    "seller_name",
    "servicer_name",
    "super_conforming_flag",
    "pre_relief_refinance_loan_sequence_number",
    "special_eligibility_program",
    "relief_refinance_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator",
]

PERFORMANCE_COLS = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_non_interest_bearing_upb",
    "ddlpi",
    "mi_recoveries",
    "net_sale_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "miscellaneous_expenses",
    "actual_loss_calculation",
    "cumulative_modification_cost",
    "interest_rate_step_indicator",
    "payment_deferral_flag",
    "estimated_loan_to_value",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost",
    "interest_bearing_upb",
]

assert len(ORIGINATION_COLS) == 32
assert len(PERFORMANCE_COLS) == 32

In [5]:
YEAR_ZIP_PATTERN = re.compile(r"historical_data_(\d{4})\.zip$", re.IGNORECASE)
QUARTER_ZIP_PATTERN = re.compile(r"historical_data_(\d{4})Q([1-4])\.zip$", re.IGNORECASE)
ORIG_TXT_PATTERN = re.compile(r"historical_data_(\d{4})Q([1-4])\.txt$", re.IGNORECASE)
PERF_TXT_PATTERN = re.compile(r"historical_data_time_(\d{4})Q([1-4])\.txt$", re.IGNORECASE)


def find_year_zip_files(root: Path) -> list[Path]:
    """
    Find yearly ZIP archives like historical_data_1999.zip.
    """
    files = sorted([p for p in root.iterdir() if p.is_file() and YEAR_ZIP_PATTERN.search(p.name)])
    if not files:
        raise FileNotFoundError(
            f"No yearly ZIP files matching historical_data_YYYY.zip found in {root}"
        )
    return files


def ensure_dirs():
    ORIGINATION_OUT.mkdir(parents=True, exist_ok=True)
    PERFORMANCE_OUT.mkdir(parents=True, exist_ok=True)
    TEMP_ROOT.mkdir(parents=True, exist_ok=True)


def output_path(dataset: str, year: int, quarter: str) -> Path:
    base = ORIGINATION_OUT if dataset == "origination" else PERFORMANCE_OUT
    out_dir = base / f"vintage_year={year}" / f"vintage_quarter={quarter}"
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"part_{year}{quarter}.parquet"


def trim_sql(expr: str) -> str:
    return f"NULLIF(TRIM({expr}), '')"


def yyyymm_to_date_sql(expr: str) -> str:
    """
    Convert YYYYMM strings to DATE (first day of month), safely.
    """
    cleaned = trim_sql(expr)
    return f"TRY_STRPTIME({cleaned} || '01', '%Y%m%d')"


def int_sql(expr: str) -> str:
    return f"TRY_CAST({trim_sql(expr)} AS INTEGER)"


def dbl_sql(expr: str) -> str:
    return f"TRY_CAST({trim_sql(expr)} AS DOUBLE)"


def str_sql(expr: str) -> str:
    return trim_sql(expr)


def build_raw_column_map(n_cols: int) -> dict[str, str]:
    return {f"c{i+1}": "VARCHAR" for i in range(n_cols)}

In [6]:
def build_origination_select(year: int, quarter: str, source_year_zip: str, source_quarter_zip: str, source_txt_name: str) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {int_sql(c(1))}  AS credit_score,
        {yyyymm_to_date_sql(c(2))} AS first_payment_date,
        {str_sql(c(3))}  AS first_time_homebuyer_flag,
        {yyyymm_to_date_sql(c(4))} AS maturity_date,
        {int_sql(c(5))}  AS msa_or_metropolitan_division,
        {int_sql(c(6))}  AS mortgage_insurance_percentage,
        {int_sql(c(7))}  AS number_of_units,
        {str_sql(c(8))}  AS occupancy_status,
        {int_sql(c(9))}  AS original_combined_loan_to_value,
        {int_sql(c(10))} AS original_debt_to_income_ratio,
        {dbl_sql(c(11))} AS original_upb,
        {int_sql(c(12))} AS original_loan_to_value,
        {dbl_sql(c(13))} AS original_interest_rate,
        {str_sql(c(14))} AS channel,
        {str_sql(c(15))} AS prepayment_penalty_mortgage_flag,
        {str_sql(c(16))} AS amortization_type,
        {str_sql(c(17))} AS property_state,
        {str_sql(c(18))} AS property_type,
        {int_sql(c(19))} AS postal_code,
        {str_sql(c(20))} AS loan_sequence_number,
        {str_sql(c(21))} AS loan_purpose,
        {int_sql(c(22))} AS original_loan_term,
        {int_sql(c(23))} AS number_of_borrowers,
        {str_sql(c(24))} AS seller_name,
        {str_sql(c(25))} AS servicer_name,
        {str_sql(c(26))} AS super_conforming_flag,
        {str_sql(c(27))} AS pre_relief_refinance_loan_sequence_number,
        {str_sql(c(28))} AS special_eligibility_program,
        {str_sql(c(29))} AS relief_refinance_indicator,
        {int_sql(c(30))} AS property_valuation_method,
        {str_sql(c(31))} AS interest_only_indicator,
        {str_sql(c(32))} AS mi_cancellation_indicator,

        {year} AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}' AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}' AS source_txt_name,
        CURRENT_TIMESTAMP AS ingested_at_utc
    """

In [8]:
def build_performance_select(year: int, quarter: str, source_year_zip: str, source_quarter_zip: str, source_txt_name: str) -> str:
    c = lambda i: f"c{i}"
    return f"""
    SELECT
        {str_sql(c(1))}  AS loan_sequence_number,
        {yyyymm_to_date_sql(c(2))} AS monthly_reporting_period,
        {dbl_sql(c(3))}  AS current_actual_upb,
        {str_sql(c(4))}  AS current_loan_delinquency_status,
        {int_sql(c(5))}  AS loan_age,
        {int_sql(c(6))}  AS remaining_months_to_legal_maturity,
        {yyyymm_to_date_sql(c(7))} AS defect_settlement_date,
        {str_sql(c(8))}  AS modification_flag,
        {str_sql(c(9))}  AS zero_balance_code,
        {yyyymm_to_date_sql(c(10))} AS zero_balance_effective_date,
        {dbl_sql(c(11))} AS current_interest_rate,
        {dbl_sql(c(12))} AS current_non_interest_bearing_upb,
        {yyyymm_to_date_sql(c(13))} AS ddlpi,
        {dbl_sql(c(14))} AS mi_recoveries,
        {str_sql(c(15))} AS net_sale_proceeds,
        {dbl_sql(c(16))} AS non_mi_recoveries,
        {dbl_sql(c(17))} AS expenses,
        {dbl_sql(c(18))} AS legal_costs,
        {dbl_sql(c(19))} AS maintenance_and_preservation_costs,
        {dbl_sql(c(20))} AS taxes_and_insurance,
        {dbl_sql(c(21))} AS miscellaneous_expenses,
        {dbl_sql(c(22))} AS actual_loss_calculation,
        {dbl_sql(c(23))} AS cumulative_modification_cost,
        {str_sql(c(24))} AS interest_rate_step_indicator,
        {str_sql(c(25))} AS payment_deferral_flag,
        {int_sql(c(26))} AS estimated_loan_to_value,
        {dbl_sql(c(27))} AS zero_balance_removal_upb,
        {dbl_sql(c(28))} AS delinquent_accrued_interest,
        {str_sql(c(29))} AS delinquency_due_to_disaster,
        {str_sql(c(30))} AS borrower_assistance_status_code,
        {dbl_sql(c(31))} AS current_month_modification_cost,
        {dbl_sql(c(32))} AS interest_bearing_upb,

        {year} AS vintage_year,
        '{quarter}' AS vintage_quarter,
        '{year}{quarter}' AS vintage,
        '{source_year_zip}' AS source_year_zip,
        '{source_quarter_zip}' AS source_quarter_zip,
        '{source_txt_name}' AS source_txt_name,
        CURRENT_TIMESTAMP AS ingested_at_utc
    """

In [9]:
def connect_duckdb(db_path: Path) -> duckdb.DuckDBPyConnection:
    con = duckdb.connect(str(db_path))
    con.execute(f"PRAGMA threads={MAX_THREADS}")
    con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute("PRAGMA enable_progress_bar")
    return con

In [10]:
def extract_member_to_temp(zf: zipfile.ZipFile, member_name: str, temp_dir: Path) -> Path:
    """
    Extract one ZIP member to a temporary file and return its path.
    """
    target = temp_dir / Path(member_name).name
    with zf.open(member_name) as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    return target

In [11]:
def ingest_txt_with_duckdb(
    con: duckdb.DuckDBPyConnection,
    txt_path: Path,
    out_path: Path,
    dataset: str,
    year: int,
    quarter: str,
    source_year_zip: str,
    source_quarter_zip: str,
    source_txt_name: str,
) -> None:
    """
    Read a Freddie TXT with DuckDB and write a single parquet file.
    """
    if out_path.exists() and not OVERWRITE_EXISTING:
        return

    n_cols = 32
    raw_columns = build_raw_column_map(n_cols)

    select_sql = (
        build_origination_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
        if dataset == "origination"
        else build_performance_select(year, quarter, source_year_zip, source_quarter_zip, source_txt_name)
    )

    raw_csv = f"""
        read_csv(
            '{txt_path.as_posix()}',
            delim='|',
            header=False,
            columns={json.dumps(raw_columns)},
            quote='',
            escape='',
            null_padding=True,
            sample_size=-1,
            ignore_errors=False
        )
    """

    copy_sql = f"""
        COPY (
            {select_sql}
            FROM {raw_csv}
        )
        TO '{out_path.as_posix()}'
        (
            FORMAT PARQUET,
            COMPRESSION {PARQUET_COMPRESSION},
            ROW_GROUP_SIZE {PARQUET_ROW_GROUP_SIZE}
        )
    """

    con.execute(copy_sql)

In [12]:
def inspect_inner_quarter_zip(inner_zip_bytes: bytes, inner_zip_name: str) -> dict:
    """
    Inspect one quarter ZIP and identify origination/performance TXT files.
    """
    result = {
        "inner_zip_name": inner_zip_name,
        "year": None,
        "quarter": None,
        "orig_txt": None,
        "perf_txt": None,
    }

    m = QUARTER_ZIP_PATTERN.search(Path(inner_zip_name).name)
    if not m:
        raise ValueError(f"Unexpected quarter ZIP name: {inner_zip_name}")

    result["year"] = int(m.group(1))
    result["quarter"] = f"Q{m.group(2)}"

    with zipfile.ZipFile(io.BytesIO(inner_zip_bytes), "r") as qzip:
        names = qzip.namelist()
        for n in names:
            base = Path(n).name
            if ORIG_TXT_PATTERN.search(base):
                result["orig_txt"] = n
            elif PERF_TXT_PATTERN.search(base):
                result["perf_txt"] = n

    if result["orig_txt"] is None or result["perf_txt"] is None:
        raise ValueError(
            f"Could not identify both TXT files inside {inner_zip_name}. "
            f"Found orig={result['orig_txt']}, perf={result['perf_txt']}"
        )

    return result

In [14]:
ensure_dirs()
year_zips = find_year_zip_files(DATA_ROOT)

manifest_rows = []

for year_zip_path in tqdm(year_zips, desc="Scanning yearly ZIPs", unit="year_zip"):
    with zipfile.ZipFile(year_zip_path, "r") as yz:
        inner_names = sorted(
            [
                n for n in yz.namelist()
                if QUARTER_ZIP_PATTERN.search(Path(n).name)
            ]
        )
        for inner_name in inner_names:
            with yz.open(inner_name) as f:
                inner_bytes = f.read()
            meta = inspect_inner_quarter_zip(inner_bytes, inner_name)

            manifest_rows.append({
                "source_year_zip": year_zip_path.name,
                "source_year_zip_path": str(year_zip_path),
                "source_quarter_zip": Path(inner_name).name,
                "year": meta["year"],
                "quarter": meta["quarter"],
                "orig_txt": Path(meta["orig_txt"]).name,
                "perf_txt": Path(meta["perf_txt"]).name,
                "orig_out": str(output_path("origination", meta["year"], meta["quarter"])),
                "perf_out": str(output_path("performance", meta["year"], meta["quarter"])),
            })

manifest = pd.DataFrame(manifest_rows).sort_values(["year", "quarter"]).reset_index(drop=True)
manifest

Scanning yearly ZIPs:   0%|          | 0/27 [00:00<?, ?year_zip/s]

,source_year_zip,source_year_zip_path,source_quarter_zip,year,quarter,orig_txt,perf_txt,orig_out,perf_out
0,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q1.zip,1999,Q1,historical_data_1999Q1.txt,historical_data_time_1999Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
1,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q2.zip,1999,Q2,historical_data_1999Q2.txt,historical_data_time_1999Q2.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
2,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q3.zip,1999,Q3,historical_data_1999Q3.txt,historical_data_time_1999Q3.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
3,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q4.zip,1999,Q4,historical_data_1999Q4.txt,historical_data_time_1999Q4.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
4,historical_data_2000.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2000Q1.zip,2000,Q1,historical_data_2000Q1.txt,historical_data_time_2000Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
...,...,...,...,...,...,...,...,...,...
102,historical_data_2024.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2024Q3.zip,2024,Q3,historical_data_2024Q3.txt,historical_data_time_2024Q3.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
103,historical_data_2024.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2024Q4.zip,2024,Q4,historical_data_2024Q4.txt,historical_data_time_2024Q4.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
104,historical_data_2025.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2025Q1.zip,2025,Q1,historical_data_2025Q1.txt,historical_data_time_2025Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
105,historical_data_2025.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2025Q2.zip,2025,Q2,historical_data_2025Q2.txt,historical_data_time_2025Q2.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....


In [15]:
display(manifest.head(12))
print(f"Quarter ZIPs found: {len(manifest)}")
print(f"Years covered: {manifest['year'].min()} - {manifest['year'].max()}")
print(manifest.groupby("year").size().tail(10))

,source_year_zip,source_year_zip_path,source_quarter_zip,year,quarter,orig_txt,perf_txt,orig_out,perf_out
0,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q1.zip,1999,Q1,historical_data_1999Q1.txt,historical_data_time_1999Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
1,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q2.zip,1999,Q2,historical_data_1999Q2.txt,historical_data_time_1999Q2.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
2,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q3.zip,1999,Q3,historical_data_1999Q3.txt,historical_data_time_1999Q3.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
3,historical_data_1999.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_1999Q4.zip,1999,Q4,historical_data_1999Q4.txt,historical_data_time_1999Q4.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
4,historical_data_2000.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2000Q1.zip,2000,Q1,historical_data_2000Q1.txt,historical_data_time_2000Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
5,historical_data_2000.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2000Q2.zip,2000,Q2,historical_data_2000Q2.txt,historical_data_time_2000Q2.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
6,historical_data_2000.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2000Q3.zip,2000,Q3,historical_data_2000Q3.txt,historical_data_time_2000Q3.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
7,historical_data_2000.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2000Q4.zip,2000,Q4,historical_data_2000Q4.txt,historical_data_time_2000Q4.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
8,historical_data_2001.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2001Q1.zip,2001,Q1,historical_data_2001Q1.txt,historical_data_time_2001Q1.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....
9,historical_data_2001.zip,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,historical_data_2001Q2.zip,2001,Q2,historical_data_2001Q2.txt,historical_data_time_2001Q2.txt,C:\Users\websi\OneDrive - UT Cloud\Semester\4....,C:\Users\websi\OneDrive - UT Cloud\Semester\4....


Quarter ZIPs found: 107
Years covered: 1999 - 2025
year
2016    4
2017    4
2018    4
2019    4
2020    4
2021    4
2022    4
2023    4
2024    4
2025    3
dtype: int64


In [16]:
def run_ingestion(manifest: pd.DataFrame) -> None:
    con = connect_duckdb(DUCKDB_PATH)

    try:
        year_groups = list(manifest.groupby("year", sort=True))

        for year, year_df in tqdm(year_groups, desc="Ingesting years", unit="year"):
            year_zip_path = Path(year_df["source_year_zip_path"].iloc[0])

            with zipfile.ZipFile(year_zip_path, "r") as year_zip:
                quarter_records = year_df.to_dict(orient="records")

                for rec in tqdm(
                    quarter_records,
                    desc=f"Year {year}",
                    unit="quarter",
                    leave=False
                ):
                    qzip_name = rec["source_quarter_zip"]
                    vintage_year = int(rec["year"])
                    vintage_quarter = rec["quarter"]

                    orig_out = Path(rec["orig_out"])
                    perf_out = Path(rec["perf_out"])

                    if (
                        orig_out.exists()
                        and perf_out.exists()
                        and not OVERWRITE_EXISTING
                    ):
                        continue

                    with year_zip.open(qzip_name) as inner_f:
                        inner_bytes = inner_f.read()

                    with zipfile.ZipFile(io.BytesIO(inner_bytes), "r") as quarter_zip:
                        temp_q_dir = TEMP_ROOT / f"{vintage_year}{vintage_quarter}"
                        temp_q_dir.mkdir(parents=True, exist_ok=True)

                        try:
                            orig_txt_tmp = extract_member_to_temp(quarter_zip, rec["orig_txt"], temp_q_dir)
                            perf_txt_tmp = extract_member_to_temp(quarter_zip, rec["perf_txt"], temp_q_dir)

                            file_bar = tqdm(
                                total=2,
                                desc=f"{vintage_year}{vintage_quarter}",
                                unit="file",
                                leave=False
                            )

                            if not (orig_out.exists() and not OVERWRITE_EXISTING):
                                ingest_txt_with_duckdb(
                                    con=con,
                                    txt_path=orig_txt_tmp,
                                    out_path=orig_out,
                                    dataset="origination",
                                    year=vintage_year,
                                    quarter=vintage_quarter,
                                    source_year_zip=rec["source_year_zip"],
                                    source_quarter_zip=rec["source_quarter_zip"],
                                    source_txt_name=rec["orig_txt"],
                                )
                            file_bar.update(1)

                            if not (perf_out.exists() and not OVERWRITE_EXISTING):
                                ingest_txt_with_duckdb(
                                    con=con,
                                    txt_path=perf_txt_tmp,
                                    out_path=perf_out,
                                    dataset="performance",
                                    year=vintage_year,
                                    quarter=vintage_quarter,
                                    source_year_zip=rec["source_year_zip"],
                                    source_quarter_zip=rec["source_quarter_zip"],
                                    source_txt_name=rec["perf_txt"],
                                )
                            file_bar.update(1)
                            file_bar.close()

                        finally:
                            shutil.rmtree(temp_q_dir, ignore_errors=True)

    finally:
        con.close()

In [17]:
run_ingestion(manifest)

Ingesting years:   0%|          | 0/27 [00:00<?, ?year/s]

Year 1999:   0%|          | 0/4 [00:00<?, ?quarter/s]

1999Q1:   0%|          | 0/2 [00:00<?, ?file/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1999Q2:   0%|          | 0/2 [00:00<?, ?file/s]

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Year 2025:   0%|          | 0/3 [00:00<?, ?quarter/s]

2025Q1:   0%|          | 0/2 [00:00<?, ?file/s]

2025Q2:   0%|          | 0/2 [00:00<?, ?file/s]

2025Q3:   0%|          | 0/2 [00:00<?, ?file/s]

In [18]:
orig_files = sorted(ORIGINATION_OUT.rglob("*.parquet"))
perf_files = sorted(PERFORMANCE_OUT.rglob("*.parquet"))

print(f"Origination parquet files: {len(orig_files)}")
print(f"Performance parquet files: {len(perf_files)}")

if orig_files:
    print("First origination file:", orig_files[0])
if perf_files:
    print("First performance file:", perf_files[0])

Origination parquet files: 107
Performance parquet files: 107
First origination file: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data\parquet\origination\vintage_year=1999\vintage_quarter=Q1\part_1999Q1.parquet
First performance file: C:\Users\websi\OneDrive - UT Cloud\Semester\4. SS2026\MA5_10 Masterarbeit (30 ECTS)\data\freddie_mac_sflld_data\parquet\performance\vintage_year=1999\vintage_quarter=Q1\part_1999Q1.parquet


In [19]:
con = connect_duckdb(DUCKDB_PATH)

orig_glob = (ORIGINATION_OUT / "**" / "*.parquet").as_posix()
perf_glob = (PERFORMANCE_OUT / "**" / "*.parquet").as_posix()

orig_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{orig_glob}', union_by_name=True)").fetchone()[0]
perf_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{perf_glob}', union_by_name=True)").fetchone()[0]

print(f"Origination rows: {orig_count:,}")
print(f"Performance rows: {perf_count:,}")

Origination rows: 48,597,637
Performance rows: 2,800,558,227


In [20]:
# Origination sample
con.execute(f"""
    SELECT
        vintage_year,
        vintage_quarter,
        COUNT(*) AS n_loans,
        AVG(original_interest_rate) AS avg_rate,
        AVG(original_upb) AS avg_upb
    FROM read_parquet('{orig_glob}', union_by_name=True)
    GROUP BY 1, 2
    ORDER BY 1, 2
    LIMIT 20
""").df()

,vintage_year,vintage_quarter,n_loans,avg_rate,avg_upb
0,1999,Q1,486627,6.789075,114305.061166
1,1999,Q2,337374,7.031179,114287.345794
2,1999,Q3,222195,7.681161,115283.854272
3,1999,Q4,171497,7.855267,117176.551193
4,2000,Q1,117437,8.172757,117383.950544
5,2000,Q2,183723,8.317632,121785.731781
6,2000,Q3,209449,8.181943,125496.879909
7,2000,Q4,235759,7.847034,130605.940812
8,2001,Q1,402947,7.000148,137037.146324
9,2001,Q2,628471,6.980615,135174.299212


In [21]:
# Performance sample
con.execute(f"""
    SELECT
        monthly_reporting_period,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT loan_sequence_number) AS n_loans
    FROM read_parquet('{perf_glob}', union_by_name=True)
    GROUP BY 1
    ORDER BY 1
    LIMIT 20
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,monthly_reporting_period,n_rows,n_loans
0,1999-01-01,173,173
1,1999-02-01,51456,51456
2,1999-03-01,174497,174497
3,1999-04-01,346259,346259
4,1999-05-01,515768,515768
5,1999-06-01,635918,635918
6,1999-07-01,706086,706086
7,1999-08-01,800882,800882
8,1999-09-01,873419,873419
9,1999-10-01,931650,931650


In [22]:
con.execute(f"""
CREATE OR REPLACE VIEW sf_origination AS
SELECT * FROM read_parquet('{orig_glob}', union_by_name=True)
""")

con.execute(f"""
CREATE OR REPLACE VIEW sf_performance AS
SELECT * FROM read_parquet('{perf_glob}', union_by_name=True)
""")

print("Views created: sf_origination, sf_performance")

Views created: sf_origination, sf_performance


In [24]:
con.execute("""
WITH o AS (
    SELECT COUNT(*) AS n_orig_rows,
           COUNT(DISTINCT loan_sequence_number) AS n_orig_loans
    FROM sf_origination
),
p AS (
    SELECT COUNT(*) AS n_perf_rows,
           COUNT(DISTINCT loan_sequence_number) AS n_perf_loans
    FROM sf_performance
),
op AS (
    SELECT COUNT(*) AS n_orig_without_perf
    FROM sf_origination o
    LEFT JOIN (
        SELECT DISTINCT loan_sequence_number
        FROM sf_performance
    ) p
    USING (loan_sequence_number)
    WHERE p.loan_sequence_number IS NULL
),
po AS (
    SELECT COUNT(*) AS n_perf_without_orig
    FROM (
        SELECT DISTINCT loan_sequence_number
        FROM sf_performance
    ) p
    LEFT JOIN sf_origination o
    USING (loan_sequence_number)
    WHERE o.loan_sequence_number IS NULL
)
SELECT *
FROM o, p, op, po
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_orig_rows,n_orig_loans,n_perf_rows,n_perf_loans,n_orig_without_perf,n_perf_without_orig
0,48597637,48597637,2800558227,48597636,1,0


In [25]:
con.execute("""
SELECT
    COUNT(*) AS total_orig_rows,
    COUNT(DISTINCT loan_sequence_number) AS distinct_loans,
    COUNT(*) - COUNT(DISTINCT loan_sequence_number) AS duplicate_rows
FROM sf_origination
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_orig_rows,distinct_loans,duplicate_rows
0,48597637,48597637,0


In [26]:
con.execute("""
SELECT
    loan_sequence_number,
    COUNT(*) AS n_rows
FROM sf_origination
GROUP BY 1
HAVING COUNT(*) > 1
ORDER BY n_rows DESC, loan_sequence_number
LIMIT 50
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,loan_sequence_number,n_rows


In [27]:
con.execute("""
SELECT * FROM (
    SELECT
        'credit_score' AS variable,
        COUNT(*) AS n,
        SUM(CASE WHEN credit_score IS NULL THEN 1 ELSE 0 END) AS n_null,
        AVG(CASE WHEN credit_score IS NULL THEN 1.0 ELSE 0.0 END) AS pct_null,
        MIN(credit_score) AS min_val,
        MAX(credit_score) AS max_val
    FROM sf_origination

    UNION ALL

    SELECT
        'original_ltv',
        COUNT(*),
        SUM(CASE WHEN original_loan_to_value IS NULL THEN 1 ELSE 0 END),
        AVG(CASE WHEN original_loan_to_value IS NULL THEN 1.0 ELSE 0.0 END),
        MIN(original_loan_to_value),
        MAX(original_loan_to_value)
    FROM sf_origination

    UNION ALL

    SELECT
        'original_dti',
        COUNT(*),
        SUM(CASE WHEN original_debt_to_income_ratio IS NULL THEN 1 ELSE 0 END),
        AVG(CASE WHEN original_debt_to_income_ratio IS NULL THEN 1.0 ELSE 0.0 END),
        MIN(original_debt_to_income_ratio),
        MAX(original_debt_to_income_ratio)
    FROM sf_origination

    UNION ALL

    SELECT
        'original_upb',
        COUNT(*),
        SUM(CASE WHEN original_upb IS NULL THEN 1 ELSE 0 END),
        AVG(CASE WHEN original_upb IS NULL THEN 1.0 ELSE 0.0 END),
        MIN(original_upb),
        MAX(original_upb)
    FROM sf_origination

    UNION ALL

    SELECT
        'original_interest_rate',
        COUNT(*),
        SUM(CASE WHEN original_interest_rate IS NULL THEN 1 ELSE 0 END),
        AVG(CASE WHEN original_interest_rate IS NULL THEN 1.0 ELSE 0.0 END),
        MIN(original_interest_rate),
        MAX(original_interest_rate)
    FROM sf_origination
)
ORDER BY variable
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,variable,n,n_null,pct_null,min_val,max_val
0,credit_score,48597637,0.0,0.000000e+00,300.0,9999.00
1,original_dti,48597637,0.0,0.000000e+00,1.0,999.00
2,original_interest_rate,48597637,1.0,2.057713e-08,0.0,13.95
3,original_ltv,48597637,0.0,0.000000e+00,1.0,999.00
4,original_upb,48597637,0.0,0.000000e+00,1000.0,2300000.00


In [28]:
con.execute("""
SELECT
    COUNT(*) AS n_loans,

    quantile_cont(credit_score, 0.01) AS cs_p01,
    quantile_cont(credit_score, 0.05) AS cs_p05,
    quantile_cont(credit_score, 0.50) AS cs_p50,
    quantile_cont(credit_score, 0.95) AS cs_p95,
    quantile_cont(credit_score, 0.99) AS cs_p99,

    quantile_cont(original_loan_to_value, 0.01) AS ltv_p01,
    quantile_cont(original_loan_to_value, 0.50) AS ltv_p50,
    quantile_cont(original_loan_to_value, 0.95) AS ltv_p95,
    quantile_cont(original_loan_to_value, 0.99) AS ltv_p99,

    quantile_cont(original_debt_to_income_ratio, 0.01) AS dti_p01,
    quantile_cont(original_debt_to_income_ratio, 0.50) AS dti_p50,
    quantile_cont(original_debt_to_income_ratio, 0.95) AS dti_p95,
    quantile_cont(original_debt_to_income_ratio, 0.99) AS dti_p99,

    quantile_cont(original_upb, 0.01) AS upb_p01,
    quantile_cont(original_upb, 0.50) AS upb_p50,
    quantile_cont(original_upb, 0.95) AS upb_p95,
    quantile_cont(original_upb, 0.99) AS upb_p99,

    quantile_cont(original_interest_rate, 0.01) AS rate_p01,
    quantile_cont(original_interest_rate, 0.50) AS rate_p50,
    quantile_cont(original_interest_rate, 0.95) AS rate_p95,
    quantile_cont(original_interest_rate, 0.99) AS rate_p99
FROM sf_origination
WHERE credit_score IS NOT NULL
   OR original_loan_to_value IS NOT NULL
   OR original_debt_to_income_ratio IS NOT NULL
   OR original_upb IS NOT NULL
   OR original_interest_rate IS NOT NULL
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_loans,cs_p01,cs_p05,cs_p50,cs_p95,cs_p99,ltv_p01,ltv_p50,ltv_p95,ltv_p99,...,dti_p95,dti_p99,upb_p01,upb_p50,upb_p95,upb_p99,rate_p01,rate_p50,rate_p95,rate_p99
0,48597637,606.0,648.0,754.0,807.0,816.0,22.0,75.0,95.0,104.0,...,999.0,999.0,40000.0,184000.0,468000.0,645000.0,2.375,4.875,7.25,8.125


In [29]:
con.execute("""
SELECT
    occupancy_status,
    property_type,
    loan_purpose,
    channel,
    COUNT(*) AS n_loans,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS pct_loans
FROM sf_origination
GROUP BY 1, 2, 3, 4
ORDER BY n_loans DESC
LIMIT 50
""").df()

,occupancy_status,property_type,loan_purpose,channel,n_loans,pct_loans
0,P,SF,N,R,8374602,17.233
1,P,SF,C,R,4993227,10.275
2,P,SF,P,R,4918357,10.121
3,P,SF,C,T,2422368,4.985
4,P,SF,P,C,2348086,4.832
5,P,SF,N,T,2300182,4.733
6,P,PU,P,R,1901534,3.913
7,P,PU,N,R,1809196,3.723
8,P,SF,P,T,1773077,3.648
9,P,SF,N,C,1664231,3.425


In [30]:
con.execute("""
SELECT 'occupancy_status' AS variable, occupancy_status AS category, COUNT(*) AS n
FROM sf_origination
GROUP BY 1, 2

UNION ALL

SELECT 'property_type', property_type, COUNT(*)
FROM sf_origination
GROUP BY 1, 2

UNION ALL

SELECT 'loan_purpose', loan_purpose, COUNT(*)
FROM sf_origination
GROUP BY 1, 2

UNION ALL

SELECT 'channel', channel, COUNT(*)
FROM sf_origination
GROUP BY 1, 2

ORDER BY variable, n DESC
""").df()

,variable,category,n
0,channel,R,27087663
1,channel,C,9134570
2,channel,T,8700993
3,channel,B,3672599
4,channel,9,1812
5,loan_purpose,N,18730680
6,loan_purpose,P,17914591
7,loan_purpose,C,11952315
8,loan_purpose,9,51
9,occupancy_status,P,44116576


In [32]:
con.execute("""
WITH loan_hist AS (
    SELECT
        loan_sequence_number,
        COUNT(*) AS n_monthly_rows,
        MIN(monthly_reporting_period) AS first_report_month,
        MAX(monthly_reporting_period) AS last_report_month,
        MIN(loan_age) AS min_loan_age,
        MAX(loan_age) AS max_loan_age
    FROM sf_performance
    GROUP BY 1
)
SELECT
    COUNT(*) AS n_loans,
    AVG(n_monthly_rows) AS avg_n_monthly_rows,
    quantile_cont(n_monthly_rows, 0.50) AS p50_n_monthly_rows,
    quantile_cont(n_monthly_rows, 0.90) AS p90_n_monthly_rows,
    quantile_cont(n_monthly_rows, 0.99) AS p99_n_monthly_rows,
    MIN(n_monthly_rows) AS min_n_monthly_rows,
    MAX(n_monthly_rows) AS max_n_monthly_rows
FROM loan_hist
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_loans,avg_n_monthly_rows,p50_n_monthly_rows,p90_n_monthly_rows,p99_n_monthly_rows,min_n_monthly_rows,max_n_monthly_rows
0,48597636,57.627458,47.0,122.0,201.0,1,320


In [31]:
con.execute("""
WITH loan_hist AS (
    SELECT
        loan_sequence_number,
        COUNT(*) AS n_monthly_rows,
        MIN(loan_age) AS min_loan_age,
        MAX(loan_age) AS max_loan_age
    FROM sf_performance
    GROUP BY 1
)
SELECT *
FROM loan_hist
WHERE n_monthly_rows <= 1
   OR min_loan_age < 0
   OR max_loan_age < min_loan_age
ORDER BY n_monthly_rows, max_loan_age
LIMIT 100
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,loan_sequence_number,n_monthly_rows,min_loan_age,max_loan_age
0,F25Q30028850,1,0,0
1,F25Q30021134,1,0,0
2,F25Q30030924,1,0,0
3,F25Q30034695,1,0,0
4,F25Q30031821,1,0,0
...,...,...,...,...
95,F25Q30058107,1,0,0
96,F25Q30037846,1,0,0
97,F25Q30033220,1,0,0
98,F25Q30027781,1,0,0


In [33]:
con.execute("""
SELECT
    current_loan_delinquency_status,
    COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_rows
FROM sf_performance
GROUP BY 1
ORDER BY n_rows DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,current_loan_delinquency_status,n_rows,pct_rows
0,0,2731257157,97.5255
1,1,30274385,1.0810
2,2,8793826,0.3140
3,3,4228346,0.1510
4,RA,3080305,0.1100
...,...,...,...
217,216,1,0.0000
218,209,1,0.0000
219,212,1,0.0000
220,217,1,0.0000


In [34]:
con.execute("""
SELECT
    CASE
        WHEN current_loan_delinquency_status IN ('0', '00') THEN 'current'
        WHEN current_loan_delinquency_status IN ('1','2','3','4','5','6') THEN '1-6 months delinquent'
        WHEN current_loan_delinquency_status IN ('RA','RM') THEN 'special delinquency code'
        WHEN current_loan_delinquency_status IS NULL THEN 'null'
        ELSE 'other'
    END AS delinquency_bucket,
    COUNT(*) AS n_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_rows
FROM sf_performance
GROUP BY 1
ORDER BY n_rows DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,delinquency_bucket,n_rows,pct_rows
0,current,2731257157,97.5255
1,1-6 months delinquent,50721904,1.8111
2,other,15498861,0.5534
3,special delinquency code,3080305,0.1100


In [35]:
con.execute("""
SELECT
    zero_balance_code,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT loan_sequence_number) AS n_loans,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_rows
FROM sf_performance
WHERE zero_balance_code IS NOT NULL
GROUP BY 1
ORDER BY n_rows DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,zero_balance_code,n_rows,n_loans,pct_rows
0,01,34278478,34278478,97.3000
1,09,362618,362618,1.0293
2,16,183829,183829,0.5218
3,03,132346,132346,0.3757
4,96,127009,127009,0.3605
5,02,107887,107887,0.3062
6,15,37528,37528,0.1065


In [36]:
con.execute("""
WITH loan_event AS (
    SELECT
        loan_sequence_number,
        MAX(CASE WHEN zero_balance_code IS NOT NULL THEN 1 ELSE 0 END) AS ever_zero_balance,
        MAX(CASE WHEN current_loan_delinquency_status NOT IN ('0', '00') AND current_loan_delinquency_status IS NOT NULL THEN 1 ELSE 0 END) AS ever_delinquent,
        MAX(CASE WHEN modification_flag = 'Y' THEN 1 ELSE 0 END) AS ever_modified
    FROM sf_performance
    GROUP BY 1
)
SELECT
    COUNT(*) AS n_loans,
    AVG(ever_zero_balance) AS share_ever_zero_balance,
    AVG(ever_delinquent) AS share_ever_delinquent,
    AVG(ever_modified) AS share_ever_modified
FROM loan_event
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_loans,share_ever_zero_balance,share_ever_delinquent,share_ever_modified
0,48597636,0.724926,0.132565,0.011563


In [37]:
con.execute("""
WITH latest_perf AS (
    SELECT *
    FROM (
        SELECT
            p.*,
            ROW_NUMBER() OVER (
                PARTITION BY loan_sequence_number
                ORDER BY monthly_reporting_period DESC
            ) AS rn
        FROM sf_performance p
    )
    WHERE rn = 1
),
loan_panel AS (
    SELECT
        o.vintage_year,
        o.vintage_quarter,
        o.loan_sequence_number,
        o.credit_score,
        o.original_loan_to_value,
        o.original_debt_to_income_ratio,
        o.original_interest_rate,
        o.original_upb,
        lp.current_actual_upb,
        lp.current_loan_delinquency_status,
        lp.zero_balance_code,
        lp.monthly_reporting_period AS latest_reporting_period
    FROM sf_origination o
    LEFT JOIN latest_perf lp
      ON o.loan_sequence_number = lp.loan_sequence_number
)
SELECT
    vintage_year,
    vintage_quarter,
    COUNT(*) AS n_loans,
    AVG(credit_score) AS avg_credit_score,
    AVG(original_loan_to_value) AS avg_orig_ltv,
    AVG(original_debt_to_income_ratio) AS avg_orig_dti,
    AVG(original_interest_rate) AS avg_orig_rate,
    AVG(original_upb) AS avg_orig_upb,
    AVG(CASE WHEN current_loan_delinquency_status NOT IN ('0', '00') AND current_loan_delinquency_status IS NOT NULL THEN 1.0 ELSE 0.0 END) AS latest_dq_share,
    AVG(CASE WHEN zero_balance_code IS NOT NULL THEN 1.0 ELSE 0.0 END) AS latest_zero_balance_share
FROM loan_panel
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (67.2 GiB/67.2 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

In [ ]:
con.execute("""
WITH latest_perf AS (
    SELECT *
    FROM (
        SELECT
            p.*,
            ROW_NUMBER() OVER (
                PARTITION BY loan_sequence_number
                ORDER BY monthly_reporting_period DESC
            ) AS rn
        FROM sf_performance p
    )
    WHERE rn = 1
)
SELECT
    o.vintage_year,
    o.vintage_quarter,
    COUNT(*) AS n_loans,
    AVG(lp.loan_age) AS avg_latest_loan_age,
    AVG(lp.remaining_months_to_legal_maturity) AS avg_remaining_months,
    AVG(CASE WHEN lp.zero_balance_code IS NULL THEN 1.0 ELSE 0.0 END) AS share_still_active
FROM sf_origination o
LEFT JOIN latest_perf lp
  ON o.loan_sequence_number = lp.loan_sequence_number
GROUP BY 1, 2
ORDER BY 1, 2
""").df()

In [ ]:
con.execute("""
WITH ever_dq AS (
    SELECT
        loan_sequence_number,
        MAX(
            CASE
                WHEN current_loan_delinquency_status NOT IN ('0', '00')
                 AND current_loan_delinquency_status IS NOT NULL
                THEN 1 ELSE 0
            END
        ) AS ever_delinquent
    FROM sf_performance
    GROUP BY 1
),
base AS (
    SELECT
        o.loan_sequence_number,
        o.credit_score,
        o.original_loan_to_value,
        e.ever_delinquent,
        NTILE(10) OVER (ORDER BY o.credit_score) AS credit_score_decile,
        NTILE(10) OVER (ORDER BY o.original_loan_to_value) AS ltv_decile
    FROM sf_origination o
    INNER JOIN ever_dq e
      ON o.loan_sequence_number = e.loan_sequence_number
    WHERE o.credit_score IS NOT NULL
      AND o.original_loan_to_value IS NOT NULL
)
SELECT
    'credit_score_decile' AS grouping,
    credit_score_decile AS bucket,
    COUNT(*) AS n_loans,
    AVG(ever_delinquent) AS ever_dq_rate
FROM base
GROUP BY 1, 2

UNION ALL

SELECT
    'ltv_decile' AS grouping,
    ltv_decile AS bucket,
    COUNT(*) AS n_loans,
    AVG(ever_delinquent) AS ever_dq_rate
FROM base
GROUP BY 1, 2
ORDER BY grouping, bucket
""").df()

In [ ]:
con.execute("""
SELECT *
FROM sf_origination
WHERE
    original_upb <= 0
    OR original_interest_rate <= 0
    OR credit_score < 300
    OR credit_score > 850
    OR original_loan_to_value < 0
    OR original_loan_to_value > 200
    OR original_debt_to_income_ratio < 0
    OR original_debt_to_income_ratio > 100
LIMIT 200
""").df()

In [ ]:
con.execute("""
SELECT *
FROM sf_performance
WHERE
    current_actual_upb < 0
    OR current_interest_rate < 0
    OR loan_age < 0
    OR remaining_months_to_legal_maturity < 0
LIMIT 200
""").df()